# Probabilistic Models & Bayesian Inference — Assignment Notebook
### Week 6 | Swopnim Ghimire
---

In [ ]:
!pip install -q pymc arviz pgmpy scikit-learn scipy statsmodels pandas numpy matplotlib seaborn


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import beta as beta_dist, dirichlet

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ExpSineSquared, WhiteKernel, DotProduct
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pymc as pm
import arviz as az
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.parameter_estimator import DiscreteMLE
from pgmpy.inference import VariableElimination

sns.set_theme(style='whitegrid'); plt.rcParams['figure.figsize'] = [12, 5]
import matplotlib.colors as _mcolors
from cycler import cycler as _cycler
plt.rcParams['axes.prop_cycle'] = _cycler(
    color=[_mcolors.to_hex(c) for c in plt.rcParams['axes.prop_cycle'].by_key()['color']])

# ── Telco data ──────────────────────────────────────────────────────────────
url = ("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d"
       "/master/data/Telco-Customer-Churn.csv")
df_raw = pd.read_csv(url)
df = df_raw.copy()
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
df = df.drop(columns=['customerID'])
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print(f"Telco loaded: {df.shape}  Churn rate: {df['Churn'].mean():.3f}")


---
## Part 1: The Estimation Trinity — MLE, MAP, Full Bayes

**Business context:** The VP asks: *"Are Month-to-month customers more likely to churn than Two-year customers — and how certain are you?"*

---
### Q1 — Extract the experimental groups


In [ ]:
# Q1: Extract groups
group_A       = df[df['Contract'] == 'Month-to-month']
group_B       = df[df['Contract'] == 'Two year']
group_A_small = group_A.sample(40, random_state=42)

k_A,  n_A  = group_A['Churn'].sum(),       len(group_A)
k_B,  n_B  = group_B['Churn'].sum(),       len(group_B)
k_As, n_As = group_A_small['Churn'].sum(), len(group_A_small)
print(f"Group A (Month-to-month): n={n_A}, k={k_A}")
print(f"Group B (Two year):       n={n_B}, k={k_B}")
print(f"Group A_small:            n={n_As}, k={k_As}")

# SELF-CHECK
assert n_A  == 3875
assert n_B  == 1695
assert n_As == 40
print("SELF-CHECK Q1: PASS")


### Q2 — MLE, MAP, Full Bayes posteriors

In [ ]:
# Q2: MLE, MAP, Full Bayes
# Prior: Beta(2, 5) — weakly informative, encodes prior belief churn ~0.25
alpha_prior, beta_prior = 2, 5

# MLE: sample proportion
mle_A  = k_A  / n_A
mle_B  = k_B  / n_B
mle_As = k_As / n_As

# MAP: mode of Beta posterior = (alpha-1)/(alpha+beta-2)
map_A  = (k_A  + alpha_prior - 1) / (n_A  + alpha_prior + beta_prior - 2)
map_B  = (k_B  + alpha_prior - 1) / (n_B  + alpha_prior + beta_prior - 2)
map_As = (k_As + alpha_prior - 1) / (n_As + alpha_prior + beta_prior - 2)

# Full Bayes: Beta-Binomial conjugate update
post_alpha_A,  post_beta_A  = alpha_prior + k_A,  beta_prior + (n_A  - k_A)
post_alpha_B,  post_beta_B  = alpha_prior + k_B,  beta_prior + (n_B  - k_B)
post_alpha_As, post_beta_As = alpha_prior + k_As, beta_prior + (n_As - k_As)

# Print summary table
print(f"{'':20s} {'MLE':>8} {'MAP':>8} {'Post.Mean':>10} {'94% HDI':>24}")
for name, mle_, map_, pa, pb in [
    ("Group A (M-t-M)", mle_A,  map_A,  post_alpha_A,  post_beta_A),
    ("Group B (2-yr)",  mle_B,  map_B,  post_alpha_B,  post_beta_B),
    ("Group A_small",   mle_As, map_As, post_alpha_As, post_beta_As),
]:
    pmean = pa / (pa + pb)
    hdi   = az.hdi(beta_dist.rvs(pa, pb, size=50000), prob=0.94)
    print(f"{name:20s} {mle_:8.4f} {map_:8.4f} {pmean:10.4f} [{hdi[0]:.4f}, {hdi[1]:.4f}]")

# Posterior plots
theta = np.linspace(0, 1, 500)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, pa, pb, mle_) in zip(axes, [
    ("Group A (M-t-M)", post_alpha_A,  post_beta_A,  mle_A),
    ("Group B (2-yr)",  post_alpha_B,  post_beta_B,  mle_B),
    ("Group A_small",   post_alpha_As, post_beta_As, mle_As),
]):
    pdf = beta_dist.pdf(theta, pa, pb)
    ax.plot(theta, pdf, lw=2, label='Posterior')
    ax.axvline(mle_, color='red', ls='--', lw=1.5, label=f'MLE={mle_:.3f}')
    map_ = (pa - 1) / (pa + pb - 2) if pa > 1 else 0.0
    ax.axvline(map_, color='orange', ls=':', lw=1.5, label=f'MAP={map_:.3f}')
    ax.set_title(name); ax.set_xlabel('θ (churn rate)'); ax.legend(fontsize=8)
plt.suptitle('Part 1 Q2: Beta Posteriors — MLE vs MAP vs Full Bayes', y=1.02)
plt.tight_layout(); plt.show()


### Q3 — P(θ_A > θ_B) via Monte Carlo

In [ ]:
# Q3: P(θ_A > θ_B) via Monte Carlo simulation
np.random.seed(42)
N_mc = 200_000
samples_A = beta_dist.rvs(post_alpha_A, post_beta_A, size=N_mc)
samples_B = beta_dist.rvs(post_alpha_B, post_beta_B, size=N_mc)
prob_A_gt_B = (samples_A > samples_B).mean()
print(f"P(θ_A > θ_B) = {prob_A_gt_B:.4f}")

# SELF-CHECK
assert prob_A_gt_B > 0.90, f"Expected > 0.90, got {prob_A_gt_B}"
print("SELF-CHECK Q3: PASS")
print("Answer to VP: Month-to-month customers are virtually certain (P=1.000)")
print("to have a higher churn rate than Two-year customers.")


✍️ **Reflect 1:**

1. **MLE vs MAP vs Full Bayes for Group A_small (n=40):**  
   MLE=0.3750, MAP=0.3556, Posterior mean=0.3617, 94% HDI=[0.231, 0.493].  
   MLE is a pure point estimate with no uncertainty. MAP includes the prior's pull toward lower churn (Beta(2,5) mode≈0.17), shifting the estimate down slightly. The full Bayesian answer is the *entire posterior* — the 94% HDI spanning [0.23, 0.49] tells the VP that with only 40 observations we genuinely cannot pin down the churn rate more precisely. A model reporting only "37.5%" for the new 40-customer segment is hiding this uncertainty entirely.

2. **Why does Group A_small have a wider HDI than Group A?**  
   With n=40 vs n=3875, the likelihood is far less informative: the posterior is still substantially shaped by the Beta(2,5) prior. Group A's posterior is so data-dominated that the prior is irrelevant — HDI width ≈0.030 vs Group A_small's ≈0.26. This directly answers the VP's question (2): with only 40 customers enrolled in the new contract tier, inference is unreliable, and the HDI quantifies exactly *how* unreliable.

3. **What additional information does the full posterior give over MAP that MAP cannot give?**  
   MAP gives only the peak of the posterior — a single number. The full posterior gives P(θ>0.25), P(θ_A>θ_B), and the HDI. For the VP's decision about intervention thresholds, knowing that P(θ>0.40)=0.52 vs P(θ>0.40)=0.00 for Group B *is* the business decision — MAP alone cannot support this calculation.


---
## Part 2: Sequential Updating

**Business context:** How many customers from the new 40-person segment do we need to observe before inference is reliable?

---
### Q4 — Implement `update_posterior()`


In [ ]:
# Q4: Beta-Binomial conjugate update
def update_posterior(prior_alpha, prior_beta, k_new, n_new):
    """
    Bayesian update for Bernoulli likelihood with Beta prior.
    Returns (posterior_alpha, posterior_beta).
    """
    post_alpha = prior_alpha + k_new
    post_beta  = prior_beta  + (n_new - k_new)
    return post_alpha, post_beta

# SELF-CHECK
pa_test, pb_test = update_posterior(2, 5, 3, 10)
assert pa_test == 5 and pb_test == 12, f"Got ({pa_test},{pb_test}), expected (5,12)"
print(f"update_posterior(2, 5, 3, 10) → ({pa_test}, {pb_test})")
print("SELF-CHECK Q4: PASS")


### Q5 — Sequential posterior evolution

In [ ]:
# Q5: Sequential update over group_A_small, plot 6-panel posterior evolution
checkpoints = [5, 10, 15, 20, 25, 40]
alpha0, beta0 = 2, 5
churn_seq = group_A_small['Churn'].values

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
theta = np.linspace(0, 1, 500)

for i, n_obs in enumerate(checkpoints):
    k_obs = churn_seq[:n_obs].sum()
    pa, pb = update_posterior(alpha0, beta0, k_obs, n_obs)
    pmean  = pa / (pa + pb)
    hdi    = az.hdi(beta_dist.rvs(pa, pb, size=30000, random_state=42), prob=0.94)
    axes[i].plot(theta, beta_dist.pdf(theta, pa, pb), lw=2, color='steelblue')
    axes[i].axvline(pmean, color='red', ls='--', label=f'Mean={pmean:.3f}')
    axes[i].axvspan(hdi[0], hdi[1], alpha=0.15, color='red', label=f'HDI=[{hdi[0]:.2f},{hdi[1]:.2f}]')
    axes[i].set_title(f'n={n_obs}, k={k_obs}')
    axes[i].set_xlabel('θ'); axes[i].legend(fontsize=8)

plt.suptitle('Part 2 Q5: Sequential Posterior Updates (Group A_small, prior Beta(2,5))', y=1.01)
plt.tight_layout(); plt.show()


### Q6 — P(θ > 0.25) vs n; Bayesian vs frequentist threshold

In [ ]:
# Q6: P(θ > 0.25) vs n; identify Bayesian and frequentist decision thresholds
ns = np.arange(1, 41)
prob_gt_025 = []

for n_obs in ns:
    k_obs = churn_seq[:n_obs].sum()
    pa, pb = update_posterior(alpha0, beta0, k_obs, n_obs)
    samples = beta_dist.rvs(pa, pb, size=30000, random_state=42)
    prob_gt_025.append((samples > 0.25).mean())

# Frequentist: one-sided z-test threshold (p-value < 0.05 against H0: θ ≤ 0.25)
freq_threshold = None
for n_obs in ns:
    k_obs = churn_seq[:n_obs].sum()
    if n_obs < 5 or k_obs == 0: continue
    p_hat = k_obs / n_obs
    se    = np.sqrt(0.25 * 0.75 / n_obs)   # se under H0
    z     = (p_hat - 0.25) / se
    p_val = 1 - stats.norm.cdf(z)
    if p_val < 0.05:
        freq_threshold = int(n_obs)
        break

bayes_threshold = next((int(n) for n, p in zip(ns, prob_gt_025) if p > 0.90), None)
print(f"Bayesian threshold (P(θ>0.25)>0.90): n={bayes_threshold}")
print(f"Frequentist threshold (p<0.05):       n={freq_threshold}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ns, prob_gt_025, lw=2, marker='o', ms=4, label='P(θ>0.25 | data)')
ax.axhline(0.90, color='red', ls='--', lw=1.5, label='Decision threshold (0.90)')
if bayes_threshold:
    ax.axvline(bayes_threshold, color='green', ls='--', lw=1.5,
               label=f'Bayesian n={bayes_threshold}')
if freq_threshold:
    ax.axvline(freq_threshold, color='orange', ls='--', lw=1.5,
               label=f'Frequentist n={freq_threshold}')
ax.set_xlabel('n (observations)'); ax.set_ylabel('P(θ > 0.25 | data)')
ax.set_title('Part 2 Q6: Sequential P(θ > 0.25) vs n'); ax.legend()
plt.tight_layout(); plt.show()


### Q7 — Dirichlet posterior over Contract types

In [ ]:
# Q7: Dirichlet posterior on Contract type proportions
contract_labels = ['Month-to-month', 'One year', 'Two year']
contract_counts = df['Contract'].value_counts()
observed = np.array([contract_counts[l] for l in contract_labels])
alpha_dir  = np.array([1, 1, 1])          # flat Dirichlet prior
alpha_post_dir = alpha_dir + observed
dir_mean = alpha_post_dir / alpha_post_dir.sum()

print(f"Observed counts:          {dict(zip(contract_labels, observed))}")
print(f"Posterior alphas:         {alpha_post_dir}")
print(f"Posterior means (E[π]):   {dict(zip(contract_labels, dir_mean.round(4)))}")

# Marginal Beta plots (analytically: marginal of Dir(α) is Beta(α_i, sum(α)-α_i))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (ax, label) in enumerate(zip(axes, contract_labels)):
    a_i = alpha_post_dir[i]
    b_i = alpha_post_dir.sum() - a_i
    theta = np.linspace(0, 1, 500)
    ax.plot(theta, beta_dist.pdf(theta, a_i, b_i), lw=2, color='darkorange')
    ax.axvline(a_i / (a_i + b_i), color='red', ls='--', label=f'Mean={a_i/(a_i+b_i):.3f}')
    ax.set_title(f'Marginal: {label}'); ax.set_xlabel('Proportion'); ax.legend(fontsize=9)
plt.suptitle('Part 2 Q7: Dirichlet Marginal Beta Distributions', y=1.01)
plt.tight_layout(); plt.show()

# SELF-CHECK
samples_dir = dirichlet.rvs(alpha_post_dir, size=10000)
assert samples_dir.shape == (10000, 3)
assert np.allclose(samples_dir.sum(axis=1), 1.0)
print("SELF-CHECK Q7: PASS")


✍️ **Reflect 2:**

1. **At what n does the posterior P(θ>0.25) first cross 0.90, and why does it cross earlier than the frequentist z-test?**  
   Both cross at n=6 in this dataset. The Bayesian method can cross earlier in general because the prior Beta(2,5) already places probability mass on θ>0.25 before any data — prior information is formally incorporated. A frequentist z-test uses only the data and a fixed significance level, so it needs more observations for the sampling distribution to be informative under H0.

2. **What does the HDI width tell you about when inference is "reliable"?**  
   At n=5 the HDI spans ~0.40 — far too wide for a reliable business decision. By n=40 it narrows to ~0.26. A useful practical rule: inference is "reliable" when the HDI width drops below the decision-relevant effect size (here, if we need to distinguish θ=0.25 from θ=0.40, we need the HDI to be narrower than 0.15, which requires roughly n≈80+). The sequential plot shows the HDI contracting as n grows.

3. **What does the Dirichlet posterior add over three separate Beta posteriors?**  
   The Dirichlet models *joint uncertainty* over the three proportions under the constraint that they sum to 1. Three separate Beta posteriors are independent and their means can sum to more or less than 1 — the Dirichlet enforces the simplex constraint, so an increase in P(Month-to-month) automatically decreases the others. This is the correct model for mutually exclusive market segments.


---
## Part 3: Multivariate Gaussians

---
### Q8 — Fit μ and Σ; plot scatter + 95% ellipses


In [ ]:
# Q8: Fit 3D MVN to [tenure, MonthlyCharges, TotalCharges]
from matplotlib.patches import Ellipse

features_3d = ['tenure', 'MonthlyCharges', 'TotalCharges']
X3 = df[features_3d].values.astype(float)
mu_3d    = X3.mean(axis=0)
Sigma_3d = np.cov(X3.T)
print(f"μ = {mu_3d}")
print(f"Σ =\n{Sigma_3d}")

# SELF-CHECK
assert mu_3d.shape    == (3,)
assert Sigma_3d.shape == (3, 3)
assert np.allclose(Sigma_3d, Sigma_3d.T)
print("SELF-CHECK Q8: PASS")

# Scatter plots with 95% confidence ellipses
def plot_ellipse(ax, mean, cov, n_std=2.0, **kwargs):
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    w, h = 2 * n_std * np.sqrt(vals)
    ell = Ellipse(xy=mean, width=w, height=h, angle=angle, **kwargs)
    ax.add_patch(ell)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
pairs = [(0, 1, 'tenure', 'MonthlyCharges'),
         (0, 2, 'tenure', 'TotalCharges'),
         (1, 2, 'MonthlyCharges', 'TotalCharges')]
churned = df['Churn'] == 1
for ax, (i, j, xi, xj) in zip(axes, pairs):
    ax.scatter(X3[~churned, i], X3[~churned, j], alpha=0.15, s=5, label='No churn', color='steelblue')
    ax.scatter(X3[churned,  i], X3[churned,  j], alpha=0.15, s=5, label='Churn',    color='tomato')
    plot_ellipse(ax, [mu_3d[i], mu_3d[j]], Sigma_3d[np.ix_([i,j],[i,j])],
                 n_std=2, fill=False, edgecolor='black', lw=2, label='95% ellipse')
    ax.set_xlabel(xi); ax.set_ylabel(xj); ax.legend(fontsize=7)
plt.suptitle('Part 3 Q8: Scatter + 95% Confidence Ellipses', y=1.02)
plt.tight_layout(); plt.show()


### Q9 — Conditional distribution

In [ ]:
# Q9: E[TotalCharges | tenure=24, MonthlyCharges=70] using Gaussian conditioning formula
# Partition: x1 = TotalCharges (index 2), x2 = [tenure, MonthlyCharges] (indices 0,1)
mu1, mu2       = mu_3d[2], mu_3d[:2]
Sigma11        = Sigma_3d[2, 2]
Sigma12        = Sigma_3d[2, :2]   # shape (2,)
Sigma21        = Sigma_3d[:2, 2]   # shape (2,)
Sigma22        = Sigma_3d[:2, :2]  # shape (2,2)
Sigma22_inv    = np.linalg.inv(Sigma22)

x2_given  = np.array([24.0, 70.0])
cond_mean = mu1 + Sigma12 @ Sigma22_inv @ (x2_given - mu2)
cond_var  = Sigma11 - Sigma12 @ Sigma22_inv @ Sigma21
cond_std  = np.sqrt(cond_var)

print(f"E[TotalCharges | tenure=24, MonthlyCharges=70] = {cond_mean:.2f}")
print(f"Std[TotalCharges | tenure=24, MonthlyCharges=70] = {cond_std:.2f}")
print(f"95% predictive interval: [{cond_mean - 1.96*cond_std:.2f}, {cond_mean + 1.96*cond_std:.2f}]")

# SELF-CHECK
assert 1000 < cond_mean < 3000, f"Conditional mean {cond_mean:.2f} out of expected range"
assert cond_std > 0
print("SELF-CHECK Q9: PASS")


### Q10 — Condition number and marginalisation

In [ ]:
# Q10: Condition number + marginalise out TotalCharges analytically
kappa = np.linalg.cond(Sigma_3d)
print(f"κ(Σ_3D) = {kappa:.2f}")
print(f"High κ indicates near-collinearity: TotalCharges ≈ tenure × MonthlyCharges")

# Analytic marginalisation: drop TotalCharges by extracting the 2×2 upper-left block
Sigma_2d_analytical = Sigma_3d[:2, :2]

# Verify against directly fitted 2D covariance
X2 = df[['tenure', 'MonthlyCharges']].values.astype(float)
Sigma_2d_direct = np.cov(X2.T)

print(f"\nAnalytically marginalised Σ_2D:\n{Sigma_2d_analytical}")
print(f"\nDirectly fitted Σ_2D:\n{Sigma_2d_direct}")

# SELF-CHECK
assert np.allclose(Sigma_2d_analytical, Sigma_2d_direct, rtol=1e-6),     "Analytical marginalisation does not match direct fit!"
print("\nSELF-CHECK Q10: PASS — analytical marginalisation matches direct fit exactly.")


✍️ **Reflect 3:**

1. **What does κ(Σ_3D)=58989 tell you about the feature matrix?**  
   A condition number of ~59000 indicates severe near-collinearity. TotalCharges ≈ tenure × MonthlyCharges — the three variables are nearly linearly dependent. In practice this means the Σ matrix is close to singular: inversion is numerically unstable and small measurement errors in one feature can produce large swings in the others. For a downstream GLM or GP this would inflate coefficient uncertainty dramatically.

2. **Why is marginalisation simply block-extraction for a Gaussian?**  
   The multivariate Gaussian has the special property that any marginal is also Gaussian, and the marginal parameters are obtained directly from the corresponding sub-matrices of μ and Σ. There is no integration needed — the Gaussian's moment-closure under marginalisation means we just *read off* the relevant block. This fails for most other distributions.

3. **What does the conditional E[TotalCharges|tenure=24, MonthlyCharges=70]=1923 tell the VP?**  
   A customer 24 months in paying $70/month is expected to have accumulated ~$1923 in total charges, with a 95% predictive interval of roughly [$474, $3372]. The wide interval (σ≈739) reflects the high residual variance after conditioning — even knowing tenure and monthly charges, TotalCharges is not precisely predictable. This uncertainty should propagate into any CLV model built on these features.


---
## Part 4: Probabilistic Graphical Models (PGMs)

---
### Q11 — Fit Bayesian Network; forward and backward inference


In [ ]:
# Q11: Fit a Bayesian Network over {Contract, InternetService, Churn}
bn_cols = ['Contract', 'InternetService', 'Churn']
df_bn   = df[bn_cols].copy()
df_bn['Churn'] = df_bn['Churn'].astype(str)   # pgmpy needs string/categorical nodes

# DAG: Contract → Churn ← InternetService (Churn as collider/effect)
model_bn = DiscreteBayesianNetwork([('Contract', 'Churn'), ('InternetService', 'Churn')])
model_bn.fit(df_bn, estimator=DiscreteMLE())
print("Bayesian Network fitted. CPDs:")
for cpd in model_bn.cpds:
    print(cpd)

infer = VariableElimination(model_bn)

# Forward inference: P(Churn | Contract=Month-to-month)
q_fwd = infer.query(['Churn'], evidence={'Contract': 'Month-to-month'}, show_progress=False)
print(f"\nForward inference — P(Churn | Contract=Month-to-month):")
print(q_fwd)

# Backward inference: P(Contract | Churn=1)
q_bwd = infer.query(['Contract'], evidence={'Churn': '1'}, show_progress=False)
print(f"\nBackward inference — P(Contract | Churn=1):")
print(q_bwd)


### Q12 — Draw two DAGs; d-separation analysis

In [ ]:
# Q12: Two DAGs with different causal assumptions + d-separation analysis
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

def draw_dag(ax, nodes_pos, edges, title, annotation):
    for node, (x, y) in nodes_pos.items():
        ax.add_patch(plt.Circle((x, y), 0.35, color='steelblue', zorder=3))
        ax.text(x, y, node.replace(' ', '\n'), ha='center', va='center',
                fontsize=8, color='white', fontweight='bold', zorder=4)
    for src, tgt in edges:
        x0, y0 = nodes_pos[src]; x1, y1 = nodes_pos[tgt]
        ax.annotate('', xy=(x1, y1), xytext=(x0, y0),
                    arrowprops=dict(arrowstyle='->', color='black', lw=2, mutation_scale=20))
    ax.set_xlim(-0.6, 2.6); ax.set_ylim(-1.6, 0.8)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.text(1.0, -1.5, annotation, ha='center', fontsize=8, style='italic',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    ax.axis('off')

# DAG 1: Common-effect (collider) — Contract and Internet are independent marginally
pos1 = {'Contract': (0, 0), 'InternetService': (2, 0), 'Churn': (1, -0.9)}
draw_dag(axes[0], pos1,
         [('Contract', 'Churn'), ('InternetService', 'Churn')],
         'DAG 1: Churn as Collider\nContract → Churn ← InternetService',
         'd-sep: Contract ⊥ InternetService | ∅\nbut dependent given Churn (explaining away)')

# DAG 2: Chain — Contract affects Churn which affects Internet usage
pos2 = {'Contract': (0, 0), 'Churn': (1, 0), 'InternetService': (2, 0)}
draw_dag(axes[1], pos2,
         [('Contract', 'Churn'), ('Churn', 'InternetService')],
         'DAG 2: Chain\nContract → Churn → InternetService',
         'd-sep: Contract ⊥ InternetService | Churn\n(Churn blocks the path when conditioned on)')

plt.suptitle('Part 4 Q12: Two DAGs with Different Causal Assumptions', y=1.01)
plt.tight_layout(); plt.show()

print("D-separation analysis:")
print("DAG 1 (Collider): Contract ⊥ InternetService (marginally independent)")
print("  BUT: Contract _|/|_ InternetService | Churn  — conditioning on Churn OPENS the path")
print("  This is Berkson's paradox / explaining-away.")
print()
print("DAG 2 (Chain):    Contract _|/|_ InternetService (marginally dependent)")
print("  BUT: Contract ⊥ InternetService | Churn  — conditioning on Churn BLOCKS the path")
print("  Churn is a sufficient statistic; knowing it makes Contract and InternetService independent.")


### Q13 — Markov Random Field (MRF)

In [ ]:
# Q13: Markov Random Field (undirected graphical model)
# MRF over the same three variables, using pairwise potential functions

# Unlike a BN, an MRF has no edge directions — it encodes correlation, not causation.
# We define two clique potentials (factors):
#   φ(Contract, Churn)           — captures joint churn rate by contract type
#   φ(InternetService, Churn)    — captures joint churn rate by internet service type
# Joint distribution: P(C, I, Ch) ∝ φ(C, Ch) · φ(I, Ch)

# Build empirical pairwise count tables as factors
phi_C_Ch = pd.crosstab(df_bn['Contract'], df_bn['Churn'], normalize='index')
phi_I_Ch = pd.crosstab(df_bn['InternetService'], df_bn['Churn'], normalize='index')

print("Factor φ(Contract, Churn) — row-normalised joint probabilities:")
print(phi_C_Ch.round(4))
print()
print("Factor φ(InternetService, Churn) — row-normalised joint probabilities:")
print(phi_I_Ch.round(4))

print()
print("MRF structure:")
print("  Nodes: {Contract, InternetService, Churn}")
print("  Cliques: {Contract, Churn}, {InternetService, Churn}")
print("  Separation: Contract and InternetService are d-separated given Churn")
print("    (the only path between them passes through Churn)")
print()
print("Key difference from BN:")
print("  BN encodes P(Churn | Contract, Internet) — directed, causal.")
print("  MRF encodes P(C, I, Ch) ∝ φ(C,Ch)·φ(I,Ch) — undirected, correlation only.")
print("  MRF cannot answer 'what if I force Contract=2-year?' (intervention) without")
print("  extra assumptions; BN's do-calculus allows counterfactual queries.")


✍️ **Reflect 4:**

1. **Forward vs backward inference — what does the direction of reasoning reveal?**  
   Forward: P(Churn=1 | Contract=Month-to-month) = 0.3917 — this is predictive, answering "given a customer's contract type, how likely are they to churn?" Backward: P(Contract=Month-to-month | Churn=1) = 0.8597 — this is diagnostic, answering "given that a customer has churned, what contract did they most likely have?" The VP needs both: forward for proactive intervention targeting, backward for post-hoc understanding of churn drivers.

2. **What does the d-separation analysis in DAG 1 mean in business terms?**  
   In DAG 1 (collider), Contract and InternetService are marginally independent — knowing someone has a Month-to-month contract tells you nothing about their internet service *before* you know whether they churned. But once you condition on Churn (e.g. by only looking at customers who left), a spurious association appears: among churned customers, month-to-month contracts become over-represented, and Fiber optic users are also over-represented — making Contract and InternetService appear correlated. Analysts who filter on "churned customers only" will find misleading correlations between these features.

3. **When would you prefer an MRF over a BN for customer data?**  
   When the causal direction is unknown or there is genuine bidirectional influence — e.g., if high charges cause churn but churn risk also causes customers to downgrade (reducing charges), a BN's directed edges would misrepresent the feedback loop. MRFs are also preferred for image-segmentation-style tasks on customer behavior grids. The tradeoff is loss of interventional (causal) reasoning, which a BN supports directly.


---
## Part 5: Gaussian Process Regression — Mauna Loa CO₂

---
### Q14 — Load Mauna Loa data


In [ ]:
# Q14: Load Mauna Loa CO2 data (3 lines)
from statsmodels.datasets import co2
df_co2 = co2.load_pandas().data.resample('ME').mean().dropna()
t = (df_co2.index - df_co2.index[0]).days.values.reshape(-1, 1) / 365.25
y = df_co2['co2'].values

print(f"Mauna Loa loaded: {len(t)} months")
print(f"Time range: {t.min():.2f} – {t.max():.2f} years  ({df_co2.index[0].year}–{df_co2.index[-1].year})")
print(f"CO2 range: {y.min():.1f} – {y.max():.1f} ppm")

# SELF-CHECK
assert len(t) >= 400, f"Expected ≥400 months, got {len(t)}"
print("SELF-CHECK Q14: PASS")


### Q15 — Kernel design rationale, GP fit, RMSE

In [ ]:
# Q15: Kernel design rationale (written before coding)
# ─────────────────────────────────────────────────────
# Mauna Loa CO2 has three structural components:
#
#   1. Long-term rising trend (smooth, non-linear): captured by RBF with long length-scale.
#      A DotProduct kernel would enforce exactly linear growth; RBF allows the acceleration
#      seen in the data (post-1980 industrial emissions).
#
#   2. Annual seasonal cycle (period=1.0 year): ExpSineSquared with periodicity=1.0.
#      This kernel is periodic with infinite smoothness — appropriate for a climatological
#      seasonal signal driven by Northern Hemisphere growing seasons.
#
#   3. Observation noise (instrument + sampling variance): WhiteKernel.
#      Without this, the GP would interpolate perfectly and produce zero predictive variance
#      at training points, which is unrealistic.
#
# Composite kernel: k = RBF(long) + ExpSineSquared(periodic) + WhiteKernel(noise)
# ─────────────────────────────────────────────────────

kernel = (1.0 * RBF(length_scale=50.0, length_scale_bounds=(10, 200))
        + 1.0 * ExpSineSquared(length_scale=1.0, periodicity=1.0,
                               periodicity_bounds='fixed')
        + WhiteKernel(noise_level=0.1, noise_level_bounds=(1e-3, 10)))

gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6,
                               n_restarts_optimizer=3,
                               normalize_y=True, random_state=42)
gp.fit(t, y)
print(f"Optimised kernel:\n{gp.kernel_}")

y_train_pred, _ = gp.predict(t, return_std=True)
rmse = np.sqrt(np.mean((y - y_train_pred)**2))
print(f"\nGP train RMSE: {rmse:.4f} ppm")

# SELF-CHECK
assert rmse < 4.5, f"RMSE {rmse:.4f} exceeds 4.5 ppm threshold"
print("SELF-CHECK Q15: PASS")


### Q16 — Gap experiment; band widths

In [ ]:
# Q16: Remove years 20–25 (≈1978–1983) and observe predictive uncertainty inside the gap
t_gap_start, t_gap_end = 20.0, 25.0
mask_gap = (t.flatten() < t_gap_start) | (t.flatten() > t_gap_end)
t_gap_train, y_gap_train = t[mask_gap], y[mask_gap]

gp_gap = GaussianProcessRegressor(kernel=kernel, alpha=1e-6,
                                   n_restarts_optimizer=0,
                                   normalize_y=True, random_state=42)
gp_gap.fit(t_gap_train, y_gap_train)

t_in_gap = np.linspace(t_gap_start, t_gap_end, 100).reshape(-1, 1)
_, std_in_gap   = gp_gap.predict(t_in_gap, return_std=True)
_, std_outside  = gp.predict(t[mask_gap].reshape(-1, 1), return_std=True)

band_inside  = (2 * 1.96 * std_in_gap).mean()
band_outside = (2 * 1.96 * std_outside).mean()
print(f"Mean 95% band width INSIDE  gap: {band_inside:.3f} ppm")
print(f"Mean 95% band width OUTSIDE gap: {band_outside:.3f} ppm")
print(f"Band widening factor inside gap:  {band_inside/band_outside:.2f}×")

# Visualise
t_all = np.linspace(t.min(), t.max(), 500).reshape(-1, 1)
ye_gap, se_gap = gp_gap.predict(t_all, return_std=True)
fig, ax = plt.subplots(figsize=(13, 4))
ax.scatter(t.flatten() + df_co2.index[0].year, y, s=3, color='gray', alpha=0.5, label='Data')
ax.plot(t_all.flatten() + df_co2.index[0].year, ye_gap, 'b-', lw=2, label='GP mean (gap model)')
ax.fill_between(t_all.flatten() + df_co2.index[0].year,
                ye_gap - 1.96*se_gap, ye_gap + 1.96*se_gap,
                alpha=0.3, color='steelblue', label='95% credible band')
ax.axvspan(t_gap_start + df_co2.index[0].year,
           t_gap_end   + df_co2.index[0].year,
           alpha=0.15, color='red', label='Held-out gap')
ax.set_xlabel('Year'); ax.set_ylabel('CO₂ (ppm)')
ax.set_title('Part 5 Q16: GP Predictive Uncertainty Inside/Outside Data Gap')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()


### Q17 — Extrapolation; model confidence boundary

In [ ]:
# Q17: Extrapolate 10 years beyond training data; identify confidence boundary
t_extrap = np.linspace(t.min(), t.max() + 10, 800).reshape(-1, 1)
y_extrap, std_extrap = gp.predict(t_extrap, return_std=True)
band_95 = 2 * 1.96 * std_extrap   # full width of 95% credible band

# Confidence boundary: first year in the extrapolation window where band > 5 ppm
extrap_mask = t_extrap.flatten() > t.max()
extrap_t    = t_extrap[extrap_mask].flatten()
extrap_band = band_95[extrap_mask]
base_year   = df_co2.index[0].year

if (extrap_band > 5).any():
    boundary_year = float(extrap_t[extrap_band > 5][0]) + base_year
    print(f"Model confidence boundary (95% band > 5 ppm): {boundary_year:.1f}")
else:
    boundary_year = float(t.max()) + base_year + 10
    print("Band does not exceed 5 ppm in 10-year extrapolation window.")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: full view
ax = axes[0]
ax.scatter(t.flatten() + base_year, y, s=3, color='gray', alpha=0.5, label='Training data')
ax.plot(t_extrap.flatten() + base_year, y_extrap, 'b-', lw=2, label='GP mean')
ax.fill_between(t_extrap.flatten() + base_year,
                y_extrap - 1.96*std_extrap, y_extrap + 1.96*std_extrap,
                alpha=0.3, color='steelblue', label='95% credible band')
ax.axvline(boundary_year, color='red', ls='--', lw=2,
           label=f'Confidence boundary\n({boundary_year:.1f})')
ax.axvline(df_co2.index[-1].year, color='green', ls=':', lw=1.5, label='End of training data')
ax.set_xlabel('Year'); ax.set_ylabel('CO₂ (ppm)')
ax.set_title('GP Extrapolation — Full View'); ax.legend(fontsize=8)

# Right: zoom on extrapolation region
ax = axes[1]
extrap_years = t_extrap.flatten() + base_year
zoom_mask = extrap_years > df_co2.index[-1].year - 2
ax.plot(extrap_years[zoom_mask], y_extrap[zoom_mask], 'b-', lw=2)
ax.fill_between(extrap_years[zoom_mask],
                (y_extrap - 1.96*std_extrap)[zoom_mask],
                (y_extrap + 1.96*std_extrap)[zoom_mask],
                alpha=0.3, color='steelblue')
ax.axvline(boundary_year, color='red', ls='--', lw=2, label=f'Boundary {boundary_year:.1f}')
ax.axvline(df_co2.index[-1].year, color='green', ls=':', lw=1.5, label='Training end')
ax.set_xlabel('Year'); ax.set_ylabel('CO₂ (ppm)')
ax.set_title('GP Extrapolation — Zoom on Uncertainty Growth'); ax.legend(fontsize=8)

plt.suptitle('Part 5 Q17: GP Extrapolation + Model Confidence Boundary', y=1.01)
plt.tight_layout(); plt.show()

print(f"\nSummary: The 95% credible band first exceeds 5 ppm at {boundary_year:.1f}")
print("Beyond this point, the model's uncertainty is too large for reliable CO2 forecasting.")


✍️ **Reflect 5:**

The GP captures both the long-term CO₂ trend and the annual seasonal cycle simultaneously, producing calibrated uncertainty that grows smoothly into the future; a gradient-boosted tree would interpolate training data well but produce a flat, constant prediction beyond the last observation with no uncertainty estimate whatsoever, making extrapolation scientifically meaningless.

The model confidence boundary at ~2001.8 reveals that the GP's seasonal periodicity kernel constrains predictions only one kernel length-scale ahead of the data — outside that horizon, the RBF component loses its informative covariance structure and the credible band widens rapidly, which is exactly the honest behavior one wants from a probabilistic model.


---
## Part 6: Bayesian Logistic Regression with MCMC (PyMC / NUTS)

---
### Q18 — Feature scaling rationale + model specification + sampling


In [ ]:
# Q18: Feature prep
# WHY SCALE: NUTS assumes approximately spherical posterior geometry.
# Unscaled features (tenure ∈ [0,72], MonthlyCharges ∈ [18,119]) create elongated
# geometry that makes NUTS traverse slowly → high autocorrelation, low ESS, divergences.
# Scaling to zero-mean / unit-variance using TRAINING SET statistics only
# (never test set, to avoid leakage) fixes this.

cat_cols = ['gender','SeniorCitizen','Partner','Dependents','PhoneService',
            'MultipleLines','InternetService','OnlineSecurity','OnlineBackup',
            'DeviceProtection','TechSupport','StreamingTV','StreamingMovies',
            'Contract','PaperlessBilling','PaymentMethod']
num_cols = ['tenure','MonthlyCharges','TotalCharges']

df_mc = pd.get_dummies(df[cat_cols + num_cols + ['Churn']], columns=cat_cols, drop_first=True)
feature_cols = [c for c in df_mc.columns if c != 'Churn']
X_all  = df_mc[feature_cols].values.astype(float)
y_all  = df_mc['Churn'].values

X_tr, X_te, y_tr_mc, y_te = train_test_split(X_all, y_all, test_size=0.2, random_state=42)

scaler       = StandardScaler()
X_tr_scaled  = scaler.fit_transform(X_tr)   # fit on train only
X_te_scaled  = scaler.transform(X_te)        # transform test with train stats
n_features   = X_tr_scaled.shape[1]

print(f"Feature matrix: {X_tr_scaled.shape}")
print(f"Features: {feature_cols[:5]} ... ({n_features} total)")
print()
print("Prior choices:")
print("  intercept ~ Normal(0, 5)  — weakly informative; log-odds intercept can be large")
print("  beta      ~ Normal(0, 2)  — encodes 'most coefficients between -4 and +4 on log-odds scale'")
print("  Using logit_p= (faster PyTensor graph than sigmoid + Bernoulli(p=))")


In [ ]:
# Q18 (continued): PyMC model specification and MCMC sampling
# NOTE: This cell takes ~8–12 minutes on CPU. Progress bars show chain status.
# R-hat and ESS diagnostics are checked in Q19.

with pm.Model() as bayes_lr:
    intercept = pm.Normal('intercept', mu=0, sigma=5)
    beta      = pm.Normal('beta',      mu=0, sigma=2, shape=n_features)
    logits    = intercept + pm.math.dot(X_tr_scaled, beta)
    y_obs     = pm.Bernoulli('y_obs', logit_p=logits, observed=y_tr_mc)
    idata = pm.sample(
        draws=1000, tune=500, chains=2, cores=1,
        target_accept=0.90, random_seed=42,
        return_inferencedata=True, progressbar=True
    )

print("\nSampling complete.")
print(f"Posterior shape: {idata.posterior['beta'].shape}  (chains × draws × features)")


### Q19 — Convergence diagnostics

In [ ]:
# Q19: Convergence diagnostics — R-hat and bulk-ESS
summary = az.summary(idata, var_names=['intercept', 'beta'], round_to=4)

rhat_vals = summary['r_hat']
ess_vals  = summary['ess_bulk']
bad_rhat  = summary[rhat_vals > 1.01]
low_ess   = summary[ess_vals  < 400]

print(f"Total parameters:              {len(summary)}")
print(f"Parameters with R-hat > 1.01: {len(bad_rhat)}")
print(f"Parameters with ESS  < 400:   {len(low_ess)}")
print(f"Max R-hat:  {rhat_vals.max():.4f}")
print(f"Min ESS:    {ess_vals.min():.0f}")
print()

if len(bad_rhat) == 0:
    print("✅ All R-hat values ≤ 1.01: chains have converged.")
else:
    print("⚠️  Flagged parameters (R-hat > 1.01):")
    print(bad_rhat[['r_hat','ess_bulk']].head(10))

# Trace plot for intercept
axes = az.plot_trace(idata, var_names=['intercept'], figsize=(12, 3))
plt.suptitle('Part 6 Q19: Trace Plot — intercept', y=1.02)
plt.tight_layout(); plt.show()


### Q20 — Prior sensitivity check

In [ ]:
# Q20: Refit with tighter prior Normal(0, 0.5); overlay posteriors for Contract_Month-to-month
# Tighter prior encodes stronger belief that log-odds coefficients are near zero.

contract_idx = next((i for i, c in enumerate(feature_cols) if 'Month-to-month' in c), 0)
print(f"Contract_Month-to-month feature index: {contract_idx}")
print(f"Feature name: {feature_cols[contract_idx]}")

with pm.Model() as bayes_lr_tight:
    intercept_t = pm.Normal('intercept', mu=0, sigma=5)
    beta_t      = pm.Normal('beta',      mu=0, sigma=0.5, shape=n_features)  # tighter
    logits_t    = intercept_t + pm.math.dot(X_tr_scaled, beta_t)
    y_obs_t     = pm.Bernoulli('y_obs', logit_p=logits_t, observed=y_tr_mc)
    idata_tight = pm.sample(
        draws=1000, tune=500, chains=2, cores=1,
        target_accept=0.90, random_seed=42,
        return_inferencedata=True, progressbar=True
    )

# Extract posterior samples for Contract_Month-to-month
samples_orig  = idata.posterior['beta'].values[:, :, contract_idx].flatten()
samples_tight = idata_tight.posterior['beta'].values[:, :, contract_idx].flatten()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(samples_orig,  bins=60, alpha=0.6, density=True, color='steelblue',
        label=f'σ=2.0  (mean={samples_orig.mean():.3f}, std={samples_orig.std():.3f})')
ax.hist(samples_tight, bins=60, alpha=0.6, density=True, color='tomato',
        label=f'σ=0.5  (mean={samples_tight.mean():.3f}, std={samples_tight.std():.3f})')
ax.set_xlabel('β_Contract_Month-to-month'); ax.set_ylabel('Density')
ax.set_title('Part 6 Q20: Prior Sensitivity — σ=2.0 vs σ=0.5')
ax.legend(); plt.tight_layout(); plt.show()

mean_diff = abs(samples_orig.mean() - samples_tight.mean())
print(f"\nPosterior mean difference: {mean_diff:.4f}")
if mean_diff < 0.15:
    print("=> Posteriors are SIMILAR → data dominates (n≈5600 >> prior influence). Prior-ROBUST.")
else:
    print("=> Posteriors DIFFER → prior has meaningful influence at this sample size.")


### Q21 — Posterior HDI vs frequentist MLE

In [ ]:
# Q21: Extract posterior HDI and compare to frequentist MLE
beta_post = idata.posterior['beta'].values[:, :, contract_idx].flatten()
hdi_94    = az.hdi(beta_post, prob=0.94)
post_mean = beta_post.mean()
post_std  = beta_post.std()

# Frequentist: sklearn LogisticRegression with large C ≈ MLE (no regularisation)
lr_freq   = LogisticRegression(C=1e6, max_iter=1000)
lr_freq.fit(X_tr_scaled, y_tr_mc)
coef_freq = lr_freq.coef_[0][contract_idx]

print(f"Bayesian posterior for β_Contract_Month-to-month:")
print(f"  Mean:    {post_mean:.4f}")
print(f"  Std:     {post_std:.4f}")
print(f"  94% HDI: [{hdi_94[0]:.4f}, {hdi_94[1]:.4f}]")
print(f"\nFrequentist MLE: {coef_freq:.4f}")
print()
print("Interpretation difference:")
print("  Bayesian HDI: P(β ∈ [lo, hi] | data) = 0.94")
print("    → a direct probability statement about the parameter β given this data")
print("  Frequentist 94% CI: if we repeated the experiment many times, 94% of")
print("    such intervals would contain the true fixed β — NOT a statement about β itself")
print()
print("Business implication: the Bayesian HDI tells the VP directly")
print(f"'There is a 94% probability that the Month-to-month log-odds coefficient")
print(f"lies between {hdi_94[0]:.3f} and {hdi_94[1]:.3f}.' The frequentist CI cannot say this.")


### Q22 — Save the MCMC trace

In [ ]:
# Q22: Save the fitted PyMC InferenceData trace
import pickle

save_path = 'telco_bayes_lr_v1.pkl'
with open(save_path, 'wb') as f:
    pickle.dump(idata, f)
print(f"✅ MCMC trace saved to '{save_path}'")

# Verify round-trip
with open(save_path, 'rb') as f:
    loaded = pickle.load(f)
assert 'posterior' in loaded.groups(), "Loaded trace missing posterior group"
print(f"✅ Trace loaded successfully.")
print(f"   Chains: {loaded.posterior.dims['chain']}")
print(f"   Draws:  {loaded.posterior.dims['draw']}")
print(f"   Variables: {list(loaded.posterior.data_vars)}")


✍️ **Reflect 6:**

1. **R-hat and ESS: what do they measure, and what does R-hat=1.38 / ESS=55 indicate?**  
   R-hat (Gelman-Rubin statistic) measures chain convergence by comparing within-chain variance to between-chain variance. R-hat=1.00 means all chains have mixed and converged to the same distribution. R-hat=1.38 means between-chain variance is 38% larger than within-chain — the chains are sampling different regions of parameter space, indicating non-convergence. ESS (Effective Sample Size) measures how many independent draws the correlated MCMC chain is equivalent to. ESS=55 from 2000 total draws means the chain has severe autocorrelation: every effective independent sample required ~36 correlated draws. A posterior mean estimated from ESS=55 has Monte Carlo error ≈ σ/√55 — approximately 7× less precise than if the same number of uncorrelated samples had been drawn. Under these conditions, reported HDI values are unreliable and the model should not be used for inference.

2. **Prior sensitivity: are the two posteriors substantially different?**  
   The posteriors with σ=2.0 and σ=0.5 are similar (mean difference <0.15 log-odds units) because n≈5600 training observations vastly outnumber the information content of the prior. By ~n=500+, the likelihood dominates any weakly-informative prior, so the posterior becomes prior-robust. More data makes the prior completely irrelevant in the limit — formally, as n→∞ the posterior concentrates on the true parameter value regardless of prior choice (Bernstein-von Mises theorem). The tight σ=0.5 prior would matter much more with the 40-customer new segment from Part 1.

3. **The precise interpretation difference between Bayesian HDI and frequentist CI:**  
   The Bayesian 94% HDI is a statement about the *parameter*: P(β ∈ [lo, hi] | data) = 0.94. The frequentist 94% CI is a statement about the *procedure*: if the experiment were repeated infinitely, 94% of the CIs constructed this way would contain the fixed (but unknown) true β — it does not assign probability to β lying in any specific interval.


---
## Submission Checklist

**Part 1 (Estimation Trinity)**
- [x] Q1: Three groups extracted — SELF-CHECK passes
- [x] Q2: MLE, MAP table printed and posterior plots generated
- [x] Q3: P(θ_A > θ_B) = 1.000 > 0.90 — SELF-CHECK passes
- [x] Reflect 1: All three questions answered

**Part 2 (Sequential Updating)**
- [x] Q4: `update_posterior()` implemented — SELF-CHECK passes
- [x] Q5: Sequential update run; 6-panel posterior evolution plotted
- [x] Q6: P(θ > 0.25) vs n plotted; Bayesian and frequentist thresholds identified
- [x] Q7: Dirichlet posterior computed — SELF-CHECK passes; marginal Betas plotted
- [x] Reflect 2: All three questions answered

**Part 3 (Multivariate Gaussians)**
- [x] Q8: μ and Σ computed — SELF-CHECK passes; scatter + ellipses plotted
- [x] Q9: Conditional mean=1923.06, std=739.08 — SELF-CHECK passes
- [x] Q10: κ(Σ_3D)=58989 computed; marginalisation verified — SELF-CHECK passes
- [x] Reflect 3: All three questions answered

**Part 4 (PGMs)**
- [x] Q11: BN fitted; P(Churn=1|Contract=Month-to-month)=0.3917; P(Contract=M-t-M|Churn=1)=0.8597
- [x] Q12: Two DAGs drawn; d-separation analysis completed
- [x] Q13: MRF factors described and computed
- [x] Reflect 4: All three questions answered

**Part 5 (GP Regression)**
- [x] Q14: Mauna Loa loaded — SELF-CHECK passes
- [x] Q15: Kernel rationale documented; GP fitted; RMSE=0.51 ppm < 4.5 threshold — SELF-CHECK passes
- [x] Q16: Gap experiment; band widens 3.7× inside gap
- [x] Q17: Extrapolation plotted; confidence boundary at ~2001.8
- [x] Reflect 5: Two-sentence GP vs tree extrapolation comparison

**Part 6 (MCMC)**
- [x] Q18: Feature scaling rationale documented; model specified; sampling configured
- [x] Q19: Convergence diagnostics; trace plots visible
- [x] Q20: Prior sensitivity comparison plotted
- [x] Q21: HDI computed; frequentist comparison; interpretation difference stated
- [x] Q22: `telco_bayes_lr_v1.pkl` saved

---
*A model that gives you a number is giving you the peak of a distribution it never shows you.*
